# Growing Neural Cellular Automata as a Radiance Field

Standalone Cells2Voxels notebook for the radiance-field variant of Cells2Pixels `growing_rf`. It does not import from, or require, a local `Cells2Pixels` checkout. The NCA, SIREN, camera, loss, and renderer code below are adapted into this notebook so the training and export path can live on its own.

The runtime export writes `radiance_manifest.json`, `radiance_weights.bin`, and `radiance_weights.json` for a future `demo/growing_radiance_fields` viewer.


In [ ]:
import copy
import json
import math
import os
import random
import shutil
import subprocess
import sys
import zipfile
from collections import OrderedDict
from contextlib import nullcontext
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == 'notebook' else NOTEBOOK_DIR
DATA_ROOT = REPO_ROOT / 'data' / 'radiance_fields'


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def autocast_context(device: torch.device, precision: torch.dtype = torch.float32):
    enabled = device.type == 'cuda' and precision == torch.float16
    if enabled:
        return torch.autocast(device_type=device.type, dtype=precision)
    return nullcontext()


device = pick_device()
print('repo root:', REPO_ROOT)
print('data root:', DATA_ROOT)
print('device:', device)


In [ ]:
@dataclass
class RFNotebookConfig:
    experiment_name: str = 'GrowingRF_Notebook'
    seed: int = 42
    precision: str = 'float32'

    # Dataset. The scene folder should contain train_camera_params.json,
    # test_camera_params.json, train/*.png, and test/*.png.
    scene_name: str = 'lego'
    scene_path: str = str(DATA_ROOT / 'lego')
    download_if_missing: bool = True
    radiance_dataset_url: str = 'https://drive.google.com/uc?id=16FJErc6aLdI5qeQAgNunAQDWywQ9Kjrs'
    radiance_extract_dir: str = str(DATA_ROOT)
    radiance_zip_path: str = str(REPO_ROOT / 'data' / 'radiance_fields.zip')
    image_size: Tuple[int, int] = (128, 128)

    # NCA, adapted from the official growing-radiance_field config.
    grid_size: Tuple[int, int, int] = (48, 48, 48)
    channels: int = 32
    fc_dim: int = 256
    noise_level: float = 0.0
    update_prob: float = 0.5
    seed_radius: int = 3
    padding: str = 'constant'
    output_type: str = 's'

    # Radiance decoder.
    hidden_features: int = 64
    hidden_layers: int = 2
    outermost_linear: bool = True
    fx: str = 'linear'
    activation: str = 'sin'
    num_frequencies: int = 1
    first_omega_0: float = 10.0
    hidden_omega_0: float = 10.0
    sh_degree: int = 0

    # Renderer. These defaults are lighter than the paper-scale config for notebook iteration.
    num_samples: int = 96
    num_fine_samples: int = 64
    perturb: bool = True
    apply_living_mask: bool = True
    voxel_bounds: Tuple[float, float] = (-1.25, 1.25)
    background_color: float = 0.0
    density_factor: float = 1.0
    color_factor: float = 1.0
    non_linearity: str = 'softplus'
    sampler_kwargs: Dict = field(default_factory=lambda: {'mode': 'stride', 'stride': 8, 'patch_size': 64})

    # Training. Raise train_steps/image_size/grid_size after smoke tests.
    train_steps: int = 1000
    pool_size: int = 64
    batch_size: int = 1
    num_views: int = 2
    step_range: Tuple[int, int] = (8, 48)
    inject_seed_interval: int = 32
    lr: float = 1e-4
    lr_decay_steps: Tuple[int, ...] = (300, 650, 850)
    lr_decay_gamma: float = 0.3
    overflow_loss_weight: float = 100.0
    l2_weight: float = 1.0
    l1_weight: float = 1.0
    lpips_weight: float = 0.0
    summary_interval: int = 50

    export_dir: str = str(REPO_ROOT / 'demo' / 'growing_radiance_fields' / 'model' / 'lego')


cfg = RFNotebookConfig()
set_seed(cfg.seed)
cfg


## Target Radiance Scene

The next cell downloads only the radiance-field dataset archive referenced by the official Cells2Pixels data script. It extracts into this project at `data/radiance_fields`, so the notebook stays independent from any external source checkout.


In [ ]:
def radiance_required_paths(scene_path):
    scene_path = Path(scene_path)
    return [
        scene_path / 'train_camera_params.json',
        scene_path / 'test_camera_params.json',
        scene_path / 'train',
        scene_path / 'test',
    ]


def missing_radiance_paths(scene_path):
    return [p for p in radiance_required_paths(scene_path) if not p.exists()]


def import_or_install_gdown():
    try:
        import gdown
    except ModuleNotFoundError:
        print('Installing gdown for Google Drive download...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gdown'])
        import gdown
    return gdown


def download_radiance_dataset(cfg, force=False, remove_zip=False):
    scene_path = Path(cfg.scene_path)
    data_dir = Path(cfg.radiance_extract_dir)
    zip_path = Path(cfg.radiance_zip_path)

    if not force and not missing_radiance_paths(scene_path):
        print('Radiance scene already present:', scene_path)
        return scene_path

    data_dir.mkdir(parents=True, exist_ok=True)
    zip_path.parent.mkdir(parents=True, exist_ok=True)

    if force or not zip_path.exists():
        gdown = import_or_install_gdown()
        print('Downloading radiance field dataset:')
        print(' ', cfg.radiance_dataset_url)
        gdown.download(cfg.radiance_dataset_url, str(zip_path), quiet=False)
    else:
        print('Using existing archive:', zip_path)

    if not zip_path.exists() or zip_path.stat().st_size == 0:
        raise RuntimeError(f'Download did not create a valid archive: {zip_path}')

    print('Extracting radiance field dataset to:', data_dir)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)

    missing = missing_radiance_paths(scene_path)
    nested_scene = data_dir / 'radiance_fields' / scene_path.name
    if missing and nested_scene.exists() and nested_scene != scene_path:
        scene_path.mkdir(parents=True, exist_ok=True)
        for child in nested_scene.iterdir():
            dst = scene_path / child.name
            if not dst.exists():
                shutil.move(str(child), str(dst))
        missing = missing_radiance_paths(scene_path)

    if remove_zip:
        zip_path.unlink(missing_ok=True)

    if missing:
        lines = '\n'.join(f' - {p}' for p in missing)
        raise FileNotFoundError(f'Radiance dataset extracted, but required scene files are still missing:\n{lines}')

    print('Radiance scene ready:', scene_path)
    return scene_path


missing = missing_radiance_paths(cfg.scene_path)
if missing and cfg.download_if_missing:
    download_radiance_dataset(cfg)
elif missing:
    print('Missing radiance scene files:')
    for p in missing:
        print(' -', p)
    print('\nSet cfg.download_if_missing = True, or run:')
    print('  download_radiance_dataset(cfg)')
else:
    print('Radiance scene ready:', Path(cfg.scene_path))


## Standalone Camera


In [ ]:
import json

import numpy as np
import torch
import torch.nn.functional as F


# From Kaolin
def generate_transformation_matrix(camera_position, look_at, camera_up_direction):
    r"""Generate transformation matrix for given camera parameters.

    Formula is :math:`\text{P_cam} = \text{P_world} * \text{transformation_mtx}`,
    with :math:`\text{P_world}` being the points coordinates padded with 1.

    Args:
        camera_position (torch.FloatTensor):
            camera positions of shape :math:`(\text{batch_size}, 3)`,
            it means where your cameras are
        look_at (torch.FloatTensor):
            where the camera is watching, of shape :math:`(\text{batch_size}, 3)`,
        camera_up_direction (torch.FloatTensor):
            camera up directions of shape :math:`(\text{batch_size}, 3)`,
            it means what are your camera up directions, generally [0, 1, 0]

    Returns:
        (torch.FloatTensor):
            The camera transformation matrix of shape :math:`(\text{batch_size}, 4, 3)`.
    """
    z_axis = camera_position - look_at
    z_axis /= z_axis.norm(dim=1, keepdim=True)
    # torch.cross don't support broadcast
    # (https://github.com/pytorch/pytorch/issues/39656)
    if camera_up_direction.shape[0] < z_axis.shape[0]:
        camera_up_direction = camera_up_direction.repeat(z_axis.shape[0], 1)
    elif z_axis.shape[0] < camera_up_direction.shape[0]:
        z_axis = z_axis.repeat(camera_up_direction.shape[0], 1)
    x_axis = torch.cross(camera_up_direction, z_axis, dim=1)
    x_axis /= x_axis.norm(dim=1, keepdim=True)
    y_axis = torch.cross(z_axis, x_axis, dim=1)
    rot_part = torch.stack([x_axis, y_axis, z_axis], dim=2)
    trans_part = -camera_position.unsqueeze(1) @ rot_part
    return torch.cat([rot_part, trans_part], dim=1)


def generate_perspective_projection(fovyangle, ratio=1.0, dtype=torch.float):
    r"""Generate perspective projection matrix for a given camera fovy angle.

    Args:
        fovyangle (float):
            field of view angle of y axis, :math:`tan(\frac{fovy}{2}) = \frac{y}{f}`.
        ratio (float):
            aspect ratio :math:`(\frac{width}{height})`. Default: 1.0.

    Returns:
        (torch.FloatTensor):
            camera projection matrix, of shape :math:`(3, 1)`.
    """
    tanfov = np.tan(fovyangle / 2.0)
    return torch.tensor([[1.0 / (ratio * tanfov)], [1.0 / tanfov], [-1]], dtype=dtype)


class PerspectiveCamera:
    """
    This class represents a batch of cameras in 3D space.
    """

    def __init__(
        self,
        fov=60.0,
        elevation: list | np.ndarray | torch.Tensor = [0.0],
        azimuth: list | np.ndarray | torch.Tensor = [0.0],
        distance: list | np.ndarray | torch.Tensor = [2.0],
        look_at: list | np.ndarray | torch.Tensor = [0.0, 0.0, 0.0],
        up_vector: list | np.ndarray | torch.Tensor = [0.0, 1.0, 0.0],
        k=1.0,
        height=1024,
        width=1024,
        bounds=(1.0, 8.0),
        device: str | torch.device = "cuda:0",
        angle_unit="degrees",
    ):
        """
        :param elevation: Elevation angles of the cameras in degrees (list or numpy array)
        :param azimuth: Azimuth angles of the cameras in degrees (list or numpy array)
        :param distance: Distances of the cameras from the origin (list or numpy array)
        :param look_at: Point the camera is looking at (shared by all cameras)
        :param up_vector: Up vector of the camera (shared by all cameras)
        :param fov: Field of view of the camera in degrees (shared by all cameras)
        :param k: k=1.0 means the camera is perspective. As k increases the camera becomes more orthographic
        :param height: Height of the camera image (Number of pixels)
        :param width: Width of the camera image (Number of pixels)
        :param bounds: Tuple of (min, max) world coordinate defining the cube that contains the grid
        :param device: PyTorch device to store the camera data

        The camera class has the following attributes:
        transform_matrix: torch tensor with shape [num_cameras, 4, 3]
        projection_matrix: torch tensor with shape [3, 1]
        position: torch tensor with shape [num_cameras, 3]
        """

        self.k = k
        with torch.no_grad():
            self.height = height
            self.width = width
            self.fov = fov
            self.bounds = bounds

            if not isinstance(elevation, torch.Tensor):
                # Making an assumption that the type of elevation, azimuth, and distance is the same
                self.elevation = torch.tensor(
                    elevation, dtype=torch.float32, device=device
                )
                self.azimuth = torch.tensor(azimuth, dtype=torch.float32, device=device)
                self.distance = (
                    torch.tensor(distance, dtype=torch.float32, device=device)
                )
                self.look_at = torch.tensor(look_at, dtype=torch.float32, device=device)
                self.up_vector = torch.tensor(
                    up_vector, dtype=torch.float32, device=device
                )
            else:
                self.elevation, self.azimuth, self.distance = (
                    elevation,
                    azimuth,
                    distance,
                )
                self.look_at, self.up_vector = look_at, up_vector

            if self.look_at.ndim == 1:
                self.look_at = self.look_at.unsqueeze(0)

            if self.up_vector.ndim == 1:
                self.up_vector = self.up_vector.unsqueeze(0)

            if angle_unit == "degrees":
                self.elevation = self.elevation * torch.pi / 180.0
                self.azimuth = self.azimuth * torch.pi / 180.0
                self.fov = self.fov * torch.pi / 180.0

            self.projection_matrix = generate_perspective_projection(
                self.fov, dtype=torch.float32
            ).to(device)
            self.projection_matrix[:2] *= self.k

            self._update_camera()

    def _update_camera(self):
        device = self.azimuth.device
        x = self.distance * torch.cos(self.elevation) * torch.cos(self.azimuth)
        y = self.distance * torch.sin(self.elevation)
        z = self.distance * torch.cos(self.elevation) * torch.sin(self.azimuth)
        self.position = torch.stack([x, y, z], dim=1)

        self.transform_matrix = generate_transformation_matrix(
            self.position, self.look_at, self.up_vector
        ).to(device)

    def rotateY(self, angle):
        self.azimuth += angle * torch.pi / 180.0
        self._update_camera()

    @staticmethod
    def generate_random_view_cameras(
        num_views, distance=2.5, max_elevation=180.0, max_azimuth=360.0, **kwargs
    ):
        azimuth = (np.random.rand(num_views) - 0.5) * max_azimuth
        elevation = (
            np.arcsin(np.random.rand(num_views) * 2.0 - 1.0) * max_elevation / np.pi
        )
        distance = np.ones(num_views) * distance

        return PerspectiveCamera(
            elevation=elevation, azimuth=azimuth, distance=distance, **kwargs
        )

    def update_trajectory(self, t, trajectory_type="orbit"):
        """
        Updates camera parameters as a function of time to create a trajectory.

        :param t: Scalar float time parameter
        :param trajectory_type: Type of trajectory ('orbit', 'spiral', 'hover', 'wave')
        """
        if trajectory_type == "orbit":
            self.azimuth = (
                torch.tensor(
                    [t * 45.0 % 360], dtype=torch.float32, device=self.azimuth.device
                )
                * torch.pi
                / 180.0
            )
            self.elevation = (
                torch.tensor([20.0], dtype=torch.float32, device=self.azimuth.device)
                * torch.pi
                / 180.0
            )

        elif trajectory_type == "spiral":
            self.azimuth = (
                torch.tensor(
                    [t * 60.0 % 360], dtype=torch.float32, device=self.azimuth.device
                )
                * torch.pi
                / 180.0
            )
            self.elevation = (
                torch.tensor(
                    [15.0 * np.sin(t)], dtype=torch.float32, device=self.azimuth.device
                )
                * torch.pi
                / 180.0
            )
            self.distance = torch.tensor(
                [2.0 + 0.5 * np.sin(t)], dtype=torch.float32, device=self.azimuth.device
            )

        elif trajectory_type == "hover":
            self.azimuth = (
                torch.tensor([30.0], dtype=torch.float32, device=self.azimuth.device)
                * torch.pi
                / 180.0
            )
            self.elevation = (
                torch.tensor(
                    [15.0 + 10.0 * np.sin(t)],
                    dtype=torch.float32,
                    device=self.azimuth.device,
                )
                * torch.pi
                / 180.0
            )

        elif trajectory_type == "wave":
            self.azimuth = (
                torch.tensor(
                    [t * 90.0 % 360], dtype=torch.float32, device=self.azimuth.device
                )
                * torch.pi
                / 180.0
            )
            self.elevation = (
                torch.tensor(
                    [10.0 * np.sin(t * 0.5)],
                    dtype=torch.float32,
                    device=self.azimuth.device,
                )
                * torch.pi
                / 180.0
            )

        else:
            raise ValueError(f"Unknown trajectory type: {trajectory_type}")

        self._update_camera()

    def sample_along_rays(
        self, num_samples: int, perturb: bool = False, voxel_bounds=None
    ):
        """Sample points along camera rays, bounded within a cube defined by voxel_bounds.

        The cube is axis-aligned, from voxel_bounds[0] to voxel_bounds[1] in x, y, and z.
        Rays that do not intersect the cube will have NaN samples.

        Returns
        -------
        sample_points : (B, H, W, N, 3)
        t_vals        : (B, H, W, N, 1)
        ray_origins   : (B, H, W, 3)
        ray_dirs      : (B, H, W, 3)
        """

        near, far = self.bounds
        device = self.position.device
        B = self.position.shape[0]
        H, W = self.height, self.width

        # 1. Pixel grid
        i, j = torch.meshgrid(
            torch.arange(H, device=device),
            torch.arange(W, device=device),
            indexing="ij",
        )

        focal = 0.5 * W / np.tan(self.fov * 0.5)

        # Perspective camera directions (normalized)
        x_cam = (j + 0.5 - 0.5 * W) / focal
        y_cam = -(i + 0.5 - 0.5 * H) / focal
        z_cam = -torch.ones_like(x_cam)
        dirs_cam = torch.stack([x_cam, y_cam, z_cam], dim=-1)
        dirs_cam = F.normalize(dirs_cam, dim=-1)

        # 2. Interpolation between perspective and orthographic
        # alpha = 1 means fully perspective, alpha = 0 means fully orthographic
        alpha = 1.0 / self.k

        # For orthographic: all rays are parallel to z_cam direction,
        # but origins shift according to pixel coordinates
        ortho_orig_cam = torch.stack([x_cam, y_cam, torch.zeros_like(x_cam)], dim=-1)
        ortho_dirs_cam = torch.tensor([0, 0, -1.0], device=device).expand_as(ortho_orig_cam)

        # Blend camera-space rays
        blended_dirs_cam = F.normalize(alpha * dirs_cam + (1 - alpha) * ortho_dirs_cam, dim=-1)
        blended_orig_cam = alpha * torch.zeros_like(ortho_orig_cam) + (1 - alpha) * ortho_orig_cam

        # 3. Transform to world space
        R = self.transform_matrix[:, :3, :]
        blended_dirs_cam = blended_dirs_cam.view(1, H, W, 3).expand(B, H, W, 3)
        blended_orig_cam = blended_orig_cam.view(1, H, W, 3).expand(B, H, W, 3)

        ray_dirs = torch.einsum("bij,bhwj->bhwi", R, blended_dirs_cam)
        ray_dirs = F.normalize(ray_dirs, dim=-1)

        ray_origins = (
            self.position.view(B, 1, 1, 3)
            + torch.einsum("bij,bhwj->bhwi", R, blended_orig_cam)
        )

        # 4. Ray-cube intersection (same as before)
        if voxel_bounds is None:
            t_start, t_end = near, far
        else:
            bounds_min, bounds_max = voxel_bounds
            bounds_min = bounds_min * 1.05
            bounds_max = bounds_max * 1.05
            bounds_min = torch.tensor(bounds_min, device=device)
            bounds_max = torch.tensor(bounds_max, device=device)

            inv_dir = 1.0 / (ray_dirs + 1e-9)
            t_min = (bounds_min - ray_origins) * inv_dir
            t_max = (bounds_max - ray_origins) * inv_dir

            t1 = torch.minimum(t_min, t_max)
            t2 = torch.maximum(t_min, t_max)
            t_near_cube, _ = torch.max(t1, dim=-1, keepdim=True)
            t_far_cube, _ = torch.min(t2, dim=-1, keepdim=True)
            hit_mask = (t_far_cube > t_near_cube) & (t_far_cube > 0)

            t_start = torch.clamp(t_near_cube, min=near, max=far)
            t_end = torch.clamp(t_far_cube, min=near, max=far)
            t_start[~hit_mask] = near
            t_end[~hit_mask] = far

        # 5. Sample along each ray
        t_vals = torch.linspace(0.0, 1.0, num_samples, device=device)
        t_vals = t_vals.view(1, 1, 1, num_samples, 1)
        t_vals = t_start.unsqueeze(-2) + (t_end - t_start).unsqueeze(-2) * t_vals

        if perturb and num_samples > 1:
            mids = 0.5 * (t_vals[..., 1:, :] + t_vals[..., :-1, :])
            upper = torch.cat([mids, t_vals[..., -1:, :]], dim=-2)
            lower = torch.cat([t_vals[..., :1, :], mids], dim=-2)
            t_rand = torch.rand_like(t_vals)
            t_vals = lower + (upper - lower) * t_rand

        # 6. Compute final 3D sample positions
        sample_points = ray_origins.unsqueeze(-2) + ray_dirs.unsqueeze(-2) * t_vals

        return sample_points, t_vals, ray_origins, ray_dirs

    @staticmethod
    def from_json(json_path: str, device: str = "cuda:0", return_keys=False):
        """Load cameras from a NeRF-style JSON and return a PerspectiveCamera instance."""
        with open(json_path, "r", encoding="utf-8") as f:
            meta = json.load(f)

        keys = sorted(meta.keys())
        R_all, t_all = [], []
        for k in keys:
            entry = meta[k]
            R_all.append(np.array(entry["extrinsic"]["rotation"], dtype=np.float32))
            t_all.append(
                np.array(entry["extrinsic"]["translation"], dtype=np.float32).flatten()
            )

        entry0 = meta[keys[0]]
        intrinsics = entry0["intrinsic"]
        H, W = intrinsics["height"], intrinsics["width"]
        focal_px = intrinsics["focal"]
        near, far = map(float, intrinsics["bounds"])

        fov = np.degrees(2 * np.arctan(0.5 * W / focal_px))

        positions = np.stack(t_all)
        dists = np.linalg.norm(positions, axis=1)
        elevs = np.degrees(np.arcsin(positions[:, 2] / dists))
        azims = np.degrees(np.arctan2(positions[:, 0], positions[:, 1]))

        camera = PerspectiveCamera(
            fov=fov,
            elevation=elevs,
            azimuth=azims,
            distance=dists,
            look_at=[0, 0, 0],
            up_vector=[0, 1, 0],
            height=H,
            width=W,
            bounds=(near, far),
            k=1.0,
            device=device,
            angle_unit="degrees",
        )

        print("Successfully loaded cameras from JSON file.")
        if return_keys:
            return camera, keys

        return camera

    def sample_batch(self, batch_size, indices=None):
        """
        Sample a random batch of cameras from the existing camera set.
        """
        N = self.transform_matrix.shape[0]
        if batch_size > N:
            raise ValueError(
                f"Batch size {batch_size} exceeds the number of cameras {N}."
            )

        if indices is None:
            indices = np.random.choice(N, batch_size, replace=False)
        else:
            batch_size = len(indices)

        if self.up_vector.shape[0] == 1:
            up_vector = self.up_vector
            look_at = self.look_at
        else:
            up_vector = self.up_vector[indices]
            look_at = self.look_at[indices]

        return PerspectiveCamera(
            fov=self.fov,
            elevation=self.elevation[indices],
            azimuth=self.azimuth[indices],
            distance=self.distance[indices],
            look_at=look_at,
            up_vector=up_vector,
            k=self.k,
            height=self.height,
            width=self.width,
            bounds=self.bounds,
            device=self.transform_matrix.device,
            angle_unit="radians",
        )

    def __repr__(self):
        return (
            f"Camera("
            f"\n\tNumber of Cameras = {self.transform_matrix.shape[0]},"
            f"\n\tElevation = {self.elevation * 180.0 / torch.pi},"
            f"\n\tAzimuth = {self.azimuth * 180.0 / torch.pi},"
            f"\n\tDistance = {self.distance},"
            f"\n\tField of View = {self.fov * 180.0 / torch.pi},"
            f"\n\tLook At = {self.look_at},"
            f"\n\tUp Vector = {self.up_vector},"
            f"\n)"
        )


## Standalone NCA


In [ ]:
import torch


def depthwise_conv(x, filters, padding="circular"):
    """filters: [filter_n, h, w, d]"""
    b, ch, h, w, d = x.shape
    y = x.reshape(b * ch, 1, h, w, d)
    y = torch.nn.functional.pad(y, [1, 1, 1, 1, 1, 1], padding)
    y = torch.nn.functional.conv3d(y, filters[:, None])
    return y.reshape(b, -1, h, w, d)


class VNCA(torch.nn.Module):
    def __init__(
        self,
        channels=12,
        fc_dim=128,
        noise_level=0.0,
        update_prob=0.5,
        device=None,
        precision=torch.float32,
        padding="circular",
    ):
        super().__init__()
        self.channels = channels
        self.update_prob = update_prob
        self.device = device
        self.perception_kernels = 5
        self.precision = precision
        self.padding = padding
        self.register_buffer(
            "noise_level", torch.tensor([noise_level], dtype=self.precision)
        )

        input_dim = (self.channels + 0) * 5
        self.w1 = torch.nn.Conv3d(input_dim, fc_dim, 1, bias=True, device=self.device)
        self.w2 = torch.nn.Conv3d(
            fc_dim, self.channels, 1, bias=False, device=self.device
        )

        torch.nn.init.xavier_normal_(self.w1.weight, gain=0.2)
        torch.nn.init.zeros_(self.w2.weight)

        with torch.no_grad():
            delta_one = torch.tensor(
                [[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]],
                device=self.device,
                dtype=self.precision,
            )
            delta_two = torch.tensor(
                [[-2.0, 0.0, 2.0], [-4.0, 0.0, 4.0], [-2.0, 0.0, 2.0]],
                device=self.device,
                dtype=self.precision,
            )
            sobel_z = torch.stack([delta_one, delta_two, delta_one]) / 2.0
            sobel_y = sobel_z.permute(0, 2, 1)
            sobel_x = sobel_z.permute(2, 1, 0)

            lap1 = torch.tensor(
                [[2.0, 3.0, 2.0], [3.0, 6.0, 3.0], [2.0, 3.0, 2.0]],
                device=self.device,
                dtype=self.precision,
            )
            lap2 = torch.tensor(
                [[3.0, 6.0, 3.0], [6.0, -88.0, 6.0], [3.0, 6.0, 3.0]],
                device=self.device,
                dtype=self.precision,
            )
            lap = torch.stack([lap1, lap2, lap1])
            lap = lap / 8.0

            ident = torch.zeros(3, 3, 3, device=self.device, dtype=self.precision)
            ident[1, 1, 1] = 1.0

            self.filters = torch.stack([ident, sobel_x, sobel_y, sobel_z, lap])

    def forward(self, s):
        b, c, h, w, d = s.shape
        z = depthwise_conv(s, self.filters, padding=self.padding)  # [b, 5 * chn, h, w]
        delta_s = self.w2(torch.relu(self.w1(z)))
        if self.update_prob < 1.0:
            update_mask = (
                torch.rand(b, 1, h, w, d, device=s.device, dtype=self.precision)
                < self.update_prob
            ).to(self.precision)
        else:
            update_mask = 1.0
        return s + delta_s * update_mask, z

    def seed(self, n, h=128, w=128, d=128):
        with torch.no_grad():
            return (
                torch.rand(
                    n, self.channels, h, w, d, device=self.device, dtype=self.precision
                )
                - 0.5
            ) * self.noise_level


class GrowingVNCA(VNCA):
    def __init__(self, seed_radius=3, **kwargs):
        super().__init__(**kwargs)
        self.seed_radius = seed_radius
        torch.nn.init.xavier_normal_(self.w1.weight, gain=0.1)
        torch.nn.init.xavier_normal_(self.w2.weight, gain=0.1)

    def forward(self, s):
        """
        Computes one step of the NCA update rule using the specified integrator.
        The alive mask is updated based on the alpha channel.
        """
        pre_life_mask = GrowingVNCA.get_living_mask(s)
        new_s, z = super().forward(s)
        post_life_mask = GrowingVNCA.get_living_mask(new_s)

        new_s = new_s * torch.logical_and(pre_life_mask, post_life_mask).to(
            self.precision
        )
        return new_s, z

    @staticmethod
    def get_living_mask(s):
        K = 3  # kernel size
        alpha = s[:, 3:4]
        return torch.nn.functional.max_pool3d(alpha, K, stride=1, padding=K // 2) > 0.1

    def seed(self, n, h=128, w=128, d=128):
        s = torch.zeros(
            n, self.channels, h, w, d, device=self.device, dtype=self.precision
        )
        r = self.seed_radius - 1
        s[
            :,
            3:,
            h // 2 - r : h // 2 + r + 1,
            w // 2 - r : w // 2 + r + 1,
            d // 2 - r : d // 2 + r + 1,
        ] = 1.0

        return s


## Standalone SIREN


In [ ]:
import torch
import numpy as np


class SineLayer(torch.nn.Module):
    # See paper sec. 3.2, final paragraph, and supplement Sec. 1.5 for discussion of omega_0.

    # If is_first=True, omega_0 is a frequency factor which simply multiplies the activations before the
    # nonlinearity. Different signals may require different omega_0 in the first layer - this is a
    # hyperparameter.

    # If is_first=False, then the weights will be divided by omega_0 so as to keep the magnitude of
    # activations constant, but boost gradients to the weight matrix (see supplement Sec. 1.5)
    activation_functions = {
        "sin": torch.sin,
        "relu": torch.relu,
        "softplus": torch.nn.Softplus(),
        "gelu": torch.nn.GELU(),
        "lrelu": torch.nn.LeakyReLU(),
        "silu": torch.nn.SiLU(),
    }

    def __init__(self, in_features, out_features, bias=True,
                 is_first=False, omega_0=10.0, activation="sin", fx="linear"):

        super().__init__()
        assert fx in ["linear", "finer", "sinh"]
        self.fx = fx
        self.omega_0 = omega_0
        self.is_first = is_first

        self.in_features = in_features
        self.linear = torch.nn.Linear(in_features, out_features, bias=bias)

        self.activation = self.activation_functions[activation]

        self.init_weights()

    def init_weights(self):
        with torch.no_grad():
            if self.is_first:
                self.linear.weight.uniform_(-1 / self.in_features,
                                            1 / self.in_features)
            else:
                self.linear.weight.uniform_(-np.sqrt(6 / self.in_features) / self.omega_0,
                                            np.sqrt(6 / self.in_features) / self.omega_0)

            if self.fx == "finer":
                self.linear.bias.uniform_(-3.0 * np.sqrt(1.0 / self.in_features),
                                          3.0 * np.sqrt(1.0 / self.in_features))

    def forward(self, input):
        x = self.linear(input)
        if self.fx == "finer":
            x = x * (1.0 + torch.abs(x))
        elif self.fx == "sinh":
            x = torch.sinh(2.0 * x)
        return self.activation(self.omega_0 * x)


class Siren(torch.nn.Module):
    def __init__(self, in_features, coord_dim=2, hidden_features=32, hidden_layers=2, out_features=3,
                 outermost_linear=False,
                 first_omega_0=10.0, hidden_omega_0=10.0, fx="linear", activation="sin", num_frequencies=1, 
                 disable=False, disable_coords=False):
        """
        fx: The preactivation function before the sin.
            "linear": The default Siren. f(x) = x https://www.vincentsitzmann.com/siren/
            "finer": f(x) = x * (1 + abs(x)) https://arxiv.org/pdf/2312.02434
            "sinh": f(x) = sinh(x) https://arxiv.org/pdf/2410.04716

        :param activation: The activation function for the MLP layers.
        :param num_frequencies: The input coordinates will be expanded using sin/cos features.
        If num_frequencies=0 only the raw coordinates will be used. The frequencies increase linearly.
        """
        super().__init__()

        self.net = []
        self.disable = disable
        self.disable_coords = disable_coords
        self.in_features = in_features
        self.num_frequencies = num_frequencies
        coord_dim = coord_dim * max(1, 2 * num_frequencies)  # Augment coordinates with sin/cos features
        self.coord_dim = coord_dim
        if disable:
            self.net.append(torch.nn.Linear(in_features, out_features))
        else:

            self.net.append(SineLayer(in_features + coord_dim, hidden_features,
                                    is_first=True, omega_0=first_omega_0, fx=fx, activation=activation))

            for i in range(hidden_layers):
                self.net.append(SineLayer(hidden_features, hidden_features,
                                        is_first=False, omega_0=hidden_omega_0, fx=fx, activation=activation))

            if outermost_linear:
                final_linear = torch.nn.Linear(hidden_features, out_features)

                with torch.no_grad():
                    final_linear.weight.uniform_(-np.sqrt(6 / hidden_features) / hidden_omega_0,
                                                np.sqrt(6 / hidden_features) / hidden_omega_0)

                self.net.append(final_linear)
            else:
                self.net.append(SineLayer(hidden_features, out_features,
                                        is_first=False, omega_0=hidden_omega_0, fx=fx, activation=activation))

        self.net = torch.nn.Sequential(*self.net)

    def forward(self, cell_states, coords):
        """
        :param cell_states: The cell states [..., feature_dim]
        :param coords: The coordinates [..., 2] or [..., 3]
        """
        N = self.num_frequencies
        if N > 0:
            aug_coords = torch.cat([coords * 1.0 * torch.pi * i for i in range(1, N + 1)], dim=-1)
            coords = torch.cat([torch.sin(aug_coords), torch.cos(aug_coords)], dim=-1)

        if self.disable_coords:
            coords = torch.zeros_like(coords)

        if self.disable:
            x = cell_states
        else:
            x = torch.cat([coords, cell_states], dim=-1)  # Concatenate coords and cell states

        output = self.net(x)
        return output


## Standalone Radiance Renderer


In [ ]:
def rsh_cart_0(xyz: torch.Tensor):
    return torch.ones(*xyz.shape[:-1], 1, device=xyz.device, dtype=xyz.dtype)

import torch
import torch.nn.functional as F


import numpy as np
from PIL import Image



class RaySampler:
    def __init__(self, HW, **kwargs):
        self.H, self.W = HW

        self.mode = kwargs.get("mode", "stride")
        assert self.mode in ['stride', 'patch']

        self.stride = kwargs.get('stride', 1)
        self.patch_size = kwargs.get('patch_size', min(32, self.H, self.W))

        self.stride_offset_x, self.stride_offset_y = None, None
        self.patch_offset_x, self.patch_offset_y = None, None

    def sample_rays(self, x: torch.Tensor):
        """
        x: Tensor Either (B, H, W, ...) when sampling a subset of rays shot from the camera
        """
        # Sample a subset of rays from the camera
        B = x.shape[0]
        if self.mode == 'stride':
            if self.stride_offset_x is None:
                self.stride_offset_x = np.random.randint(low=0, high=self.stride, size=B)
                self.stride_offset_y = np.random.randint(low=0, high=self.stride, size=B)

            return torch.stack([
                x[i, self.stride_offset_x[i]::self.stride, self.stride_offset_y[i]::self.stride]
                for i in range(B)
            ], dim=0)

        elif self.mode == 'patch':
            if self.patch_offset_x is None:
                self.patch_offset_x = np.random.randint(low=0, high=self.H - self.patch_size + 1, size=B)
                self.patch_offset_y = np.random.randint(low=0, high=self.W - self.patch_size + 1, size=B)

            return torch.stack([
                x[i, self.patch_offset_x[i]:self.patch_offset_x[i] + self.patch_size,
                self.patch_offset_y[i]:self.patch_offset_y[i] + self.patch_size]
                for i in range(B)
            ], dim=0)

    def sample_pixels(self, x: torch.Tensor):
        """
        x: Tensor with shape (B, C, H, W) or
            with shape (B, num_views, C, H, W) when sampling a subset of pixels from the rendered image.
        """
        B = x.shape[0]
        if self.mode == 'stride':
            if self.stride_offset_x is None:
                self.stride_offset_x = np.random.randint(low=0, high=self.stride, size=B)
                self.stride_offset_y = np.random.randint(low=0, high=self.stride, size=B)

            return torch.stack([
                x[i, ..., self.stride_offset_x[i]::self.stride, self.stride_offset_y[i]::self.stride]
                for i in range(B)
            ], dim=0)

        elif self.mode == 'patch':
            if self.patch_offset_x is None:
                self.patch_offset_x = np.random.randint(low=0, high=self.H - self.patch_size + 1, size=B)
                self.patch_offset_y = np.random.randint(low=0, high=self.W - self.patch_size + 1, size=B)

            return torch.stack([
                x[i, ..., self.patch_offset_x[i]:self.patch_offset_x[i] + self.patch_size,
                self.patch_offset_y[i]:self.patch_offset_y[i] + self.patch_size]
                for i in range(B)
            ], dim=0)

def get_living_mask(s):
    K = 3  # kernel size
    alpha = s[..., 3:4]
    return torch.nn.functional.max_pool3d(alpha, K, stride=1, padding=K // 2) > 0.1

class RendererRF:
    def __init__(self, num_samples: int = 64, perturb: bool = False,
                 voxel_bounds: tuple = (-1.0, 1.0), background_color=1.0,
                 density_factor=4.0, color_factor=2.0,
                 apply_living_mask: bool = False,
                 max_rays_per_chunk: int = 16384,  # For large voxel grids, limit the number of rays processed at once
                 non_linearity: str = "softplus",  # The non-linearity applied to get a positive-value density
                 sampler_kwargs: dict = None,
                 num_fine_samples: int = 0,
                 sh_degree=0,
                 precision: torch.dtype = torch.float32,
                 **kwargs,  # Just for backward compatibility, ignore
                 ):
        self.num_samples = num_samples
        self.perturb = perturb
        self.voxel_bounds = voxel_bounds
        self.background_color = background_color
        self.density_factor = density_factor
        self.color_factor = color_factor
        self.apply_living_mask = apply_living_mask
        self.max_rays_per_chunk = max_rays_per_chunk
        self.non_linearity = {
            "softplus": F.softplus,
            "relu": F.relu,
            "sigmoid": torch.sigmoid,
        }[non_linearity]
        self.sampler_kwargs = sampler_kwargs if sampler_kwargs is not None else {}
        self.num_fine_samples = num_fine_samples
        self.sh_degree = sh_degree
        self.precision = precision

        if sh_degree != 0:
            raise NotImplementedError('Standalone notebook supports sh_degree=0, matching the official growing_rf config.')
        self.rsh_func = rsh_cart_0

    @staticmethod
    def trilinear_sample_voxel(voxel_grid: torch.Tensor,
                               sample_points: torch.Tensor,
                               bounds: tuple = (-1.0, 1.0)):
        """Trilinearly interpolate voxel_grid at the query locations.

        Args:
            voxel_grid (B, C, H, W, D): Feature grid.
            sample_points (B, h, w, N, 3): Query positions in world coordinates.
            bounds: Tuple of (min, max) world coordinate defining the cube that contains the grid.

        Returns:
            features (B, h, w, N, C): Interpolated features.
            rel_coords (B, h, w, N, 3): Relative within voxel coordinates in [-1, 1].
        """
        B, C, H, W, D = voxel_grid.shape
        min_bound, max_bound = bounds

        # Map world coordinates to normalised [-1, 1] cube expected by grid_sample.
        sample_points = 2.0 * (sample_points - min_bound) / (max_bound - min_bound) - 1.0  # still (B, h, w, N, 3)

        if sample_points.shape[0] == 1:
            sample_points = sample_points.expand(B, -1, -1, -1, -1)  # Expand to match voxel_grid batch size

        # Perform interpolation.
        # https://discuss.pytorch.org/t/surprising-convention-for-grid-sample-coordinates/79997/3
        sampled_features = F.grid_sample(voxel_grid.permute(0, 1, 4, 3, 2),  # (B, C, H, W, D) to (B, C, D, W, H)
                                         sample_points,
                                         mode="bilinear",
                                         align_corners=False)
        # sampled_features has shape (B, C, H, W, N)
        sampled_features = sampled_features.permute(0, 2, 3, 4, 1)  # (B, h, w, N, C)

        # Compute within cell relative coordinates in [0, 1]
        idx = (sample_points + 1.0) / 2.0  # Normalize to [0, 1]
        idx = idx * torch.tensor([W - 1, H - 1, D - 1], device=sample_points.device)
        relative_coords = (idx - idx.floor()) * 2.0 - 1.0  # Convert to [-1, 1] range

        return sampled_features, relative_coords

    # Hierarchical sampling from NeRF
    @staticmethod
    @torch.no_grad()
    def sample_pdf(bins, weights, N_samples, det=False):
        # Get pdf
        weights = weights + 1e-5  # prevent nans
        pdf = weights / torch.sum(weights, -1, keepdim=True)
        cdf = torch.cumsum(pdf, -1)
        cdf = torch.cat([torch.zeros_like(cdf[..., :1]), cdf], -1)  # (batch, len(bins))

        # Take uniform samples
        if det:
            u = torch.linspace(0., 1., steps=N_samples, device=bins.device)
            u = u.expand(list(cdf.shape[:-1]) + [N_samples])
        else:
            u = torch.rand(list(cdf.shape[:-1]) + [N_samples], device=bins.device)

        # Invert CDF
        u = u.contiguous()
        inds = torch.searchsorted(cdf, u, right=True)
        below = torch.max(torch.zeros_like(inds - 1), inds - 1)
        above = torch.min((cdf.shape[-1] - 1) * torch.ones_like(inds), inds)
        inds_g = torch.stack([below, above], -1)  # (batch, N_samples, 2)

        # cdf_g = tf.gather(cdf, inds_g, axis=-1, batch_dims=len(inds_g.shape)-2)
        # bins_g = tf.gather(bins, inds_g, axis=-1, batch_dims=len(inds_g.shape)-2)
        matched_shape = [inds_g.shape[0], inds_g.shape[1], cdf.shape[-1]]
        cdf_g = torch.gather(cdf.unsqueeze(1).expand(matched_shape), 2, inds_g)
        bins_g = torch.gather(bins.unsqueeze(1).expand(matched_shape), 2, inds_g)

        denom = (cdf_g[..., 1] - cdf_g[..., 0])
        denom = torch.where(denom < 1e-5, torch.ones_like(denom), denom)
        t = (u - cdf_g[..., 0]) / denom
        samples = bins_g[..., 0] + t * (bins_g[..., 1] - bins_g[..., 0])

        return samples

    def render(self,
               voxel_grid: torch.Tensor,
               camera: PerspectiveCamera,
               siren: Siren,
               batchify_rays: bool = False,
               num_samples: int = None,
               perturb: bool = None,
               voxel_bounds: tuple = None,
               background_color=None,
               density_factor=None,
               color_factor=None,
               apply_living_mask: bool = None,
               sampler_kwargs: dict = None,
               num_fine_samples: int = None,
               ):
        """Render an RGB image from *voxel_grid* using *camera* and *siren*.

        Parameters
        ----------
        voxel_grid : (B,C,H,W,D) tensor  batched feature grids
        camera     : PerspectiveCamera instance with n_view different cameras
        siren      : neural renderer f(V, x)(rgb, )
        num_samples: int  points per ray
        perturb    : bool  stratified depth sampling like NeRF
        voxel_bounds: tuple of (min, max) world coordinates defining the cube that contains the grid
        background_color: float  0.0 for black, 1.0 for white
        density_factor: float  scaling factor for the occupancy density
        color_factor: float  scaling factor for the RGB output
        apply_living_mask: bool  if True, multiply the occupancy density by the living mask (features[..., 3:4])
        sampler_kwargs: dict  additional arguments for the ray sampling function

        Returns
        -------
        rgb   : (B,num_views,3,h,w)
        depth : (B,num_views,1,h,w)
        opacity: (B,num_views,1,h,w) This is not differentiable w.r.t the living mask.
        opacity_living: (B,num_views,h,w) if apply_living_mask is True. This is differentiable w.r.t the living mask.
        sampler: RaySampler instance used for sampling the rays (deterministic sampling given the instance).
        """
        B, C, H, W, D = voxel_grid.shape
        num_views = camera.position.shape[0]
        # voxel_grid = voxel_grid.repeat(num_views, 1, 1, 1, 1)  # (B*num_views, C, H, W, D)
        voxel_grid = torch.repeat_interleave(voxel_grid, num_views, dim=0)  # (B*num_views, C, H, W, D)

        num_samples = num_samples or self.num_samples
        perturb = perturb if perturb is not None else self.perturb
        voxel_bounds = voxel_bounds or self.voxel_bounds
        background_color = background_color if background_color is not None else self.background_color
        density_factor = density_factor if density_factor is not None else self.density_factor
        color_factor = color_factor if color_factor is not None else self.color_factor
        apply_living_mask = apply_living_mask if apply_living_mask is not None else self.apply_living_mask
        sampler_kwargs = sampler_kwargs if sampler_kwargs is not None else self.sampler_kwargs
        num_fine_samples = num_fine_samples if num_fine_samples is not None else self.num_fine_samples

        def render_ray_chunk(spc, sdc, roc, rdc):
            """
            spc: sample points chunk (B*num_views, h, w, N, 3)
            sdc: sample depths chunk (B*num_views, h, w, N, 1)
            roc: ray origins chunk (B*num_views, h, w, 3)
            rdc: ray directions chunk (B*num_views, h, w, 3)
            """
            _, h, w, N, _ = spc.shape  # (B*num_views, h, w, N, 3)
            low, high = voxel_bounds

            sh_evals = self.rsh_func(rdc)  # (B*num_views, h, w, (sh_degree+1)**2)
            sh_evals = sh_evals.unsqueeze(-2) # (B*num_views, h, w, 1, (sh_degree+1)**2)
            sh_evals = sh_evals.repeat_interleave(3, -1) # (B*num_views, h, w, 1, 3*(sh_degree+1)**2)

            deltas = sdc[..., 1:, :] - sdc[..., :-1, :]
            last_delta = 1e10 * torch.ones_like(deltas[..., :1, :])
            deltas = torch.cat([deltas, last_delta], dim=-2)  # (B*num_views, h, w, N, 1)

            # 2. voxel coarse lookup
            if num_fine_samples > 0 and apply_living_mask:
                inside = ((spc >= low) & (spc <= high)).all(dim=-1).unsqueeze(-1)  # (B*num_views, h, w, N, 1)
                living_mask, relative_coords = self.trilinear_sample_voxel(voxel_grid[:, 3:4], spc, voxel_bounds)
                living_mask = torch.relu(living_mask)
                living_mask = living_mask * inside.float()  # Mask out points outside the voxel bounds
                alpha_living = 1.0 - torch.exp(-living_mask * deltas)
                transmittance_living = torch.cumprod(torch.cat(
                    [torch.ones_like(alpha_living[..., :1, :]), 1. - alpha_living + 1e-10], dim=-2),
                    dim=-2)[..., :-1, :]
                weights_living = (alpha_living * transmittance_living)  # (B*num_views, h, w, N, 1)

                sdc_midpoint = 0.5 * (sdc[..., 1:, 0] + sdc[..., :-1, 0])

                sdc_midpoint = sdc_midpoint.reshape(-1, N - 1)
                weights_living = weights_living.squeeze(-1)[..., 1:-1].reshape(-1, N - 2)
                fine_samples_depth = self.sample_pdf(sdc_midpoint, weights_living,
                                                     num_fine_samples, det=not perturb)  # [B*num_views*h*w, N_fine]
                fine_samples_depth = fine_samples_depth.reshape(-1, h, w, num_fine_samples)
                fine_samples_depth = fine_samples_depth.unsqueeze(-1)  # (B*num_views, h, w, N_fine, 1)

                N = N + num_fine_samples

                sdc, _ = torch.sort(torch.cat([sdc, fine_samples_depth], dim=-2), dim=-2)
                spc = sdc * rdc[..., None, :] + roc[..., None, :]

                deltas = sdc[..., 1:, :] - sdc[..., :-1, :]
                last_delta = 1e10 * torch.ones_like(deltas[..., :1, :])
                deltas = torch.cat([deltas, last_delta], dim=-2)  # (B*num_views, h, w, N, 1)



            living_mask_voxel = (torch.nn.functional.max_pool3d(voxel_grid[:, 3:4], 3, stride=1, padding=1) > 0.1).float()
            voxel_aug = torch.cat([living_mask_voxel, voxel_grid], dim=1)

            features, relative_coords = self.trilinear_sample_voxel(voxel_aug, spc, voxel_bounds)
            features, binary_living_mask = features[..., 1:], features[..., :1]
            # features: (B*num_views, h, w, N, C)
            inside = ((spc >= low) & (spc <= high)).all(dim=-1).unsqueeze(-1)  # (B*num_views, h, w, N, 1)

            # 3. neural renderer
            # autocast (fp16) is scoped to the SIREN call only; the volumetric integration below
            # stays in fp32 for numerical stability (exp / cumprod / weighted sums over samples).
            with autocast_context(features.device, self.precision):
                raw = siren(features, relative_coords)  # (B*num_views, h, w, N, 4)

            sh_coeffs = raw[..., :-1] # (B*num_views, h, w, N, 3*(sh_degree+1)**2)
            sh_colors = sh_coeffs * color_factor * sh_evals
            r = sh_colors[..., 0::3].sum(dim=-1, keepdim=True)
            g = sh_colors[..., 1::3].sum(dim=-1, keepdim=True)
            b = sh_colors[..., 2::3].sum(dim=-1, keepdim=True)
            rgb = torch.cat([r, g, b], dim=-1) # (B*num_views, h, w, N, 3)
            rgb = torch.relu(rgb)
            # rgb = torch.sigmoid(raw[..., :3] * color_factor)  # (B*num_views, h, w, N, 3)
            sigma = self.non_linearity(raw[..., 3:4]) * density_factor  # (B*num_views, h, w, N, 1) occupancy density
            #     sigma = F.softplus(raw[..., 3:4]) * 1.0  # (B*num_views, h, w, N, 1) occupancy density
            sigma = sigma * inside.float()  # Mask out points outside the voxel bounds
            if apply_living_mask:
                sigma = sigma * (binary_living_mask > 0.1).float()
                living_mask = torch.relu(features[..., 3:4])
                living_mask = living_mask * inside.float()  # Mask out points outside the voxel bounds

            # 4. volumetric integration (hierarchical alpha compositing)

            alpha = 1.0 - torch.exp(-sigma * deltas)
            transmittance = torch.cumprod(torch.cat(
                [torch.ones_like(alpha[..., :1, :]), 1. - alpha + 1e-10], dim=-2),
                dim=-2)[..., :-1, :]
            weights = (alpha * transmittance)  # (B,H,W,N,1)

            rgb_map = (weights * rgb).sum(dim=-2)  # (B*num_views,H,W,3)
            depth_map = (weights * sdc).sum(dim=-2)  # (B*num_views,H,W,1)
            acc_map = weights.sum(dim=-2)  # (B*num_views,H,W,1)

            rgb_final = rgb_map + (1. - acc_map) * background_color

            # print(depth_map.shape, weights.shape)
            # depth_map = 1.0 / torch.max(1e-6 * torch.ones_like(depth_map), depth_map / weights.sum(dim=-2).clamp(1e-6))

            rgb_final = rgb_final.view(B, num_views, h, w, 3).permute(0, 1, 4, 2, 3)  # (B, num_views, 3, h, w)
            depth_map = depth_map.view(B, num_views, h, w, 1).permute(0, 1, 4, 2, 3)  # (B, num_views, 1, h, w)
            acc_map = acc_map.view(B, num_views, h, w, 1).permute(0, 1, 4, 2, 3)  # (B, num_views, 1, h, w)

            acc_map_living = None
            if apply_living_mask:
                alpha_living = 1.0 - torch.exp(-living_mask * deltas)
                transmittance_living = torch.cumprod(torch.cat(
                    [torch.ones_like(alpha_living[..., :1, :]), 1. - alpha_living + 1e-10], dim=-2),
                    dim=-2)[..., :-1, :]
                weights_living = (alpha_living * transmittance_living)  # (B*num_views, h, w, N, 1)
                acc_map_living = weights_living.sum(dim=-2)  # (B*num_views,H,W,1)

                # (B, num_views, 1, h, w)
                acc_map_living = acc_map_living.view(B, num_views, h, w, 1).permute(0, 1, 4, 2, 3)

            return rgb_final, depth_map, acc_map, acc_map_living

        # 1. sample points along rays

        (sample_points, sample_depths,
         ray_origins, ray_dirs) = camera.sample_along_rays(num_samples=num_samples, perturb=perturb, voxel_bounds=voxel_bounds)
        
        sampler = RaySampler(sample_points.shape[1:3], **sampler_kwargs)
        sample_points = sampler.sample_rays(sample_points)  # (num_views, h, w, N, 3)
        sample_depths = sampler.sample_rays(sample_depths)  # (num_views, h, w, N, 1)
        sample_points = sample_points.repeat(B, 1, 1, 1, 1)  # (B*num_views, h, w, N, 3)
        sample_depths = sample_depths.repeat(B, 1, 1, 1, 1)  # (B*num_views, h, w, N, 1)
        ray_origins = sampler.sample_rays(ray_origins)  # (num_views, h, w, 3)
        ray_dirs = sampler.sample_rays(ray_dirs)  # (num_views, h, w, 3)
        ray_origins = ray_origins.repeat(B, 1, 1, 1)  # (B*num_views, h, w, 3)
        ray_dirs = ray_dirs.repeat(B, 1, 1, 1)  # (B*num_views, h, w, 3)

        if not batchify_rays:
            return *render_ray_chunk(sample_points, sample_depths, ray_origins, ray_dirs), sampler

        _, h, w, N, _ = sample_points.shape
        sample_points = sample_points.reshape(B * num_views, h * w, 1, N, 3)  # (B*num_views, h*w, 1, N, 3)
        sample_depths = sample_depths.reshape(B * num_views, h * w, 1, N, 1)  # (B*num_views, h*w, 1, N, 1)
        ray_origins = ray_origins.reshape(B * num_views, h * w, 1, 3)
        ray_dirs = ray_dirs.reshape(B * num_views, h * w, 1, 3)

        rgb_chunks, depth_chunks, acc_chunks, acc_living_chunks = [], [], [], []
        for i in range(0, h * w, self.max_rays_per_chunk):
            end = min(i + self.max_rays_per_chunk, h * w)
            spc = sample_points[:, i:end, :, :, :]
            sdc = sample_depths[:, i:end, :, :, :]
            roc = ray_origins[:, i:end, :, :]
            rdc = ray_dirs[:, i:end, :, :]
            rgb_chunk, depth_chunk, acc_chunk, acc_living_chunk = render_ray_chunk(spc, sdc, roc, rdc)
            rgb_chunks.append(rgb_chunk)
            depth_chunks.append(depth_chunk)
            acc_chunks.append(acc_chunk)
            acc_living_chunks.append(acc_living_chunk)

        rgb = torch.cat(rgb_chunks, dim=3).view(B, num_views, 3, h, w)
        depth = torch.cat(depth_chunks, dim=3).view(B, num_views, 1, h, w)
        opacity = torch.cat(acc_chunks, dim=3).view(B, num_views, 1, h, w)
        if apply_living_mask:
            opacity_living = torch.cat(acc_living_chunks, dim=3).view(B, num_views, 1, h, w)
        else:
            opacity_living = None

        return rgb, depth, opacity, opacity_living, sampler

    @staticmethod
    @torch.no_grad()
    def to_pil(rendered_features, batch_stack='vertical', view_stack='vertical', target_stack='horizontal'):
        """
        :param rendered_features: Tuple of (rgb, depth, occupancy, opacity_living)
        rgb: (B, num_views, 3, h, w)
        depth: (B, num_views, 1, h, w) or None
        opacity: (B, num_views, 1, h, w) or None
        opacity_living: (B, num_views, h, w) or None

        :param batch_stack: Whether to stack the batch elements vertically or horizontally.
        :param view_stack: Whether to stack the views vertically or horizontally.
        :param target_stack: Whether to stack the target channels vertically or horizontally.

        :return: A PIL Image showing the rendered images.
        """
        stack_batch = np.vstack if batch_stack == 'vertical' else np.hstack
        stack_view = np.vstack if view_stack == 'vertical' else np.hstack
        stack_target = np.vstack if target_stack == 'vertical' else np.hstack

        if len(rendered_features) == 1:
            rgb, depth, opacity, opacity_living = rendered_features[0], None, None, None
        else:
            rgb, depth, opacity, opacity_living = rendered_features

        assert rgb.dim() == 5, "The input tensor should have 5 dimensions"
        batch_size, num_views, _, height, width = rgb.shape

        image_list = []
        for x in [rgb, depth, opacity, opacity_living]:
            if x is None:
                continue

            x = x.clip(0.0, 1.0)  # Ensure values are in [0, 1]

            if x.shape[2] == 1:
                x = x.expand(-1, -1, 3, -1, -1)
            x = (x.permute(0, 1, 3, 4, 2).cpu().numpy() * 255).astype(np.uint8)
            image_list.append(
                stack_batch(
                    [stack_view(x[i]) for i in range(batch_size)]
                )  # [batch_size*height, num_views*width, num_features]
            )

        image = Image.fromarray(stack_target(image_list))
        return image


## Standalone Radiance Loss


In [ ]:
def imread(path, max_size=None, mode=None, center_crop=False):
    if isinstance(max_size, int):
        max_size = (max_size, max_size)

    img = Image.open(path)
    img = ImageOps.exif_transpose(img)

    if max_size is not None:
        img.thumbnail(max_size, Image.LANCZOS)

    if center_crop:
        width, height = img.size
        side = min(width, height)
        left = (width - side) // 2
        top = (height - side) // 2
        img = img.crop((left, top, left + side, top + side))
    else:
        width, height = img.size
        if width > height:
            pad = (0, (width - height) // 2, 0, (width - height + 1) // 2)
            img = ImageOps.expand(img, pad, fill=(255, 255, 255, 0))
        elif height > width:
            pad = ((height - width) // 2, 0, (height - width + 1) // 2, 0)
            img = ImageOps.expand(img, pad, fill=(255, 255, 255, 0))

    if max_size is not None and img.size != tuple(max_size):
        img = img.resize(tuple(max_size), Image.LANCZOS)

    if mode is not None:
        img = img.convert(mode)
    return np.asarray(img, dtype=np.float32) / 255.0


import os

from PIL import Image, ImageOps
import numpy as np
import torch



class RadianceFieldLoss(torch.nn.Module):
    """
    For now we only support square images.

    :param scene_path: Path to the scene directory containing camera parameters and images.
    :param image_size: The input image will be resized to this size.
    :param l2_weight: Weight for the L2 loss component.
    :param l1_weight: Weight for the L1 loss component.
    :param device: Torch device to use for the loss computation (e.g., 'cuda:0' or 'cpu').
    """

    def __init__(self, scene_path, image_size=(256, 256), l2_weight=1.0, l1_weight=1.0,
                 lpips_weight=0.0,
                 device='cuda:0'):
        # Consider adding the functionality to sample only parts of the rays for each image
        # (if there are speed and performacne issues and we can't increase the number of samples per ray).
        super(RadianceFieldLoss, self).__init__()
        self.scene_path = scene_path
        self.image_size = image_size
        self.l2_weight = l2_weight
        self.l1_weight = l1_weight
        self.lpips_weight = lpips_weight
        self.device = device

        self.train_cameras, self.train_images_np, self.train_images, self.train_image_names = self.load_targets(
            tag="train")
        self.test_cameras, self.test_images_np, self.test_images, self.test_image_names = self.load_targets(tag="test")

        print("Successfully loaded target images.")

        # self.target_image_pil = Image.fromarray(np.uint8(self.target_image_np * 255))

        if self.lpips_weight > 0:
            from lpips import LPIPS
            self.lpips_loss_fn = LPIPS(net='vgg').to(self.device)

    def load_targets(self, tag="train"):
        cameras, image_names = PerspectiveCamera.from_json(
            os.path.join(self.scene_path, f'{tag}_camera_params.json'),
            self.device, return_keys=True)

        cameras.height = self.image_size[0]
        cameras.width = self.image_size[1]

        valid_idx = []
        image_list = []
        for i, image_name in enumerate(image_names):
            image_path = os.path.join(self.scene_path, f'{tag}', f'{image_name}')
            if os.path.exists(image_path):
                valid_idx.append(i)
                image_list.append(imread(image_path, max_size=self.image_size, mode='RGBA', center_crop=False))

        image_names = list(np.array(image_names)[valid_idx])
        cameras = cameras.sample_batch(len(valid_idx), valid_idx)

        if not image_list:
            raise FileNotFoundError(f'No {tag} images found in {os.path.join(self.scene_path, tag)}')

        images_np = np.array(image_list)
        images_np[..., 3:4] = (images_np[..., 3:4] > 0.5) * 1.0
        images_np[..., :3] *= images_np[..., 3:4]
        images_torch = torch.tensor(images_np, dtype=torch.float32,
                                    device=self.device).permute(0, 3, 1, 2)

        return cameras, images_np, images_torch, image_names

    def get_random_views(self, batch_size=1, mode='train'):
        assert mode in ['train', 'test'], "Mode must be 'train' or 'test'."
        cameras = self.train_cameras if mode == 'train' else self.test_cameras
        images = self.train_images if mode == 'train' else self.test_images

        view_idx = np.random.choice(len(images), batch_size, replace=True)
        cameras = cameras.sample_batch(batch_size, view_idx)

        return cameras, view_idx

    @staticmethod
    @torch.no_grad()
    def calculate_psnr(img1, img2, max_val=1.0):
        """
        Computes the PSNR (Peak Signal-to-Noise Ratio) between two images.

        Args:
            img1 (Tensor): First image tensor of shape [B, C, H, W].
            img2 (Tensor): Second image tensor of shape [B, C, H, W].
            max_val (float): Maximum possible pixel value of the images (1.0 for normalized, 255 for raw).

        Returns:
            Tensor: PSNR value (scalar tensor if batch size is 1, else shape [B]).
        """
        img1 = torch.clip(img1, 0.0, 1.0)
        img2 = torch.clip(img2, 0.0, 1.0)
        mse = torch.mean((img1 - img2) ** 2)
        psnr = 20 * np.log10(max_val) - 10 * torch.log10(mse + 1e-8)
        return psnr

    def forward(self, input_dict, return_summary=True):
        """
        Calculate the image loss based on the input dictionary.
        :param input_dict: A dictionary containing 'generated_images' key with a batch of images.
                           The generated images should be in the shape of [batch_size, 4, height, width].
                           The generated images should normally be in range [0, 1].
        :param return_summary: If True, returns a summary of the loss.
        :return: Loss value and optionally a summary.
        """
        B, num_views, _, h, w = input_dict['generated_images_l1l2'].shape
        view_idx = input_dict['view_idx']
        mode = input_dict['mode']

        assert mode in ["train", "test"]
        if mode == 'train':
            target_images = self.train_images[view_idx]  # [num_views, 4, H, W]
        else:
            target_images = self.test_images[view_idx]  # [num_views, 4, H, W]

        l1l2_sampler = input_dict.get('sampler_l1l2', RaySampler(target_images.shape[2:]))
        lpips_sampler = input_dict.get('sampler_lpips', l1l2_sampler)

        target_images_l1l2 = l1l2_sampler.sample_pixels(target_images)  # [num_views, 4, h, w]
        target_images_l1l2 = target_images_l1l2.unsqueeze(0).expand(B, -1, -1, -1, -1)  # [B, num_views, 4, h, w]
        target_images_l1l2 = target_images_l1l2.reshape(B * num_views, 4, h, w)  # Flatten batch and views

        generated_images_l1l2 = input_dict['generated_images_l1l2']  # [B, num_views 4, h, w]
        generated_images_l1l2 = generated_images_l1l2.reshape(B * num_views, 4, h, w)  # Flatten batch and views

        alpha_l1l2 = input_dict['alpha_l1l2'] * 2.0  # [B, num_views, 1, h, w]

        alpha_l1l2 = alpha_l1l2.reshape(B * num_views, 1, h, w)  # Flatten batch and views

        loss = 0.0
        loss_log = {}
        if self.lpips_weight > 0:
            target_images_lpips = lpips_sampler.sample_pixels(target_images)  # [num_views, 4, h, w]
            target_images_lpips = target_images_lpips.unsqueeze(0).expand(B, -1, -1, -1, -1)  # [B, num_views, 4, h, w]
            target_images_lpips = target_images_lpips.reshape(B * num_views, 4, h, w)  # Flatten batch and views

            generated_images_lpips = input_dict.get('generated_images_lpips',
                                                    generated_images_l1l2)  # [B, num_views, 4, h, w]
            generated_images_lpips = generated_images_lpips.reshape(B * num_views, 4, h, w)

            alpha_lpips = input_dict.get('alpha_lpips', alpha_l1l2)  # [B, num_views, 1, h, w]
            alpha_lpips = alpha_lpips.reshape(B * num_views, 1, h, w)  # Flatten batch and views

            # chn = np.random.choice([0, 1, 2, 3], size=3, replace=False)
            # Sample random channels for each element in the batch
            # chn = torch.stack([torch.randperm(4, device=self.device)[:3] for _ in range(B * num_views)], dim=0)
            # x = generated_images_lpips[torch.arange(B * num_views)[:, None], chn]
            # y = target_images_lpips[torch.arange(B * num_views)[:, None], chn]

            x = generated_images_lpips[:, :3, :, :]  # Use only RGB channels for LPIPS
            y = target_images_lpips[:, :3, :, :]  # Use only RGB channels for LPIPS

            lpips_loss = self.lpips_loss_fn(x, y, normalize=True).mean()
            loss += lpips_loss * self.lpips_weight
            loss_log['LPIPS'] = lpips_loss

            generated_images_l1l2 = torch.cat([generated_images_l1l2, generated_images_lpips], dim=0)
            target_images_l1l2 = torch.cat([target_images_l1l2, target_images_lpips], dim=0)
            alpha_l1l2 = torch.cat([alpha_l1l2, alpha_lpips], dim=0)

        l2_loss = ((generated_images_l1l2 - target_images_l1l2) ** 2).mean()
        l1_loss = torch.abs(generated_images_l1l2 - target_images_l1l2).mean()
        psnr = RadianceFieldLoss.calculate_psnr(generated_images_l1l2, target_images_l1l2)

        shape_l2_loss = ((target_images_l1l2[:, 3:4] - alpha_l1l2) ** 2).mean()
        shape_l1_loss = torch.abs(target_images_l1l2[:, 3:4] - alpha_l1l2).mean()

        loss_log['L2'] = l2_loss
        loss_log['L1'] = l1_loss
        loss_log['PSNR'] = psnr
        loss_log['Shape L2'] = shape_l2_loss
        loss_log['Shape L1'] = shape_l1_loss

        loss += l2_loss * self.l2_weight + l1_loss * self.l1_weight
        loss += shape_l2_loss + shape_l1_loss  # Shape loss is not weighted

        summary = None
        if return_summary:
            summary = {
                "images_l1l2": visualize_differences(target_images_l1l2, generated_images_l1l2),
            }
            if self.lpips_weight > 0:
                summary["images_lpips"] = visualize_differences(target_images_lpips, generated_images_lpips)

        return loss, loss_log, summary


@torch.no_grad()
def visualize_differences(target: torch.Tensor, generated: torch.Tensor):
    """
    Visualizes the differences between a target image and a batch of generated images.

    Args:
        target: Tensor of shape [1, 4, H, W] (RGBA image)
        generated: Tensor of shape [B, 4, H, W] (batch of RGBA images)
    """
    assert target.shape[1] == 4 and generated.shape[1] == 4, "Images must be RGBA"
    assert target.shape[2:] == generated.shape[2:], "Target and generated images must be same size"

    generated = generated[:4]  # Limit to first 4 images for visualization
    target = target[:4]

    b, _, h, w = generated.shape
    target = torch.clip(target, 0.0, 1.0)
    generated = torch.clip(generated, 0.0, 1.0)

    # Split into RGB and alpha
    target_rgb, target_alpha = target[:, :3], target[:, 3:]
    generated_rgb, generated_alpha = generated[:, :3], generated[:, 3:]
    union_alpha = torch.clip(target_alpha + generated_alpha, 0, 1.0)

    # Compute absolute differences
    diff_rgb = torch.abs(target_rgb - generated_rgb)
    diff_alpha = torch.abs(target_alpha - generated_alpha)

    diff_rgb = torch.clip(diff_rgb, 0.0, 1.0)
    diff_alpha = torch.clip(diff_alpha, 0.0, 1.0)

    # Concatenate for visualization: [B, 4, H, W] each, so final shape will be [B, 4*W, H]

    # Convert diff_alpha to RGBA for visualization (gray to RGBA)
    diff_alpha_rgb = diff_alpha.expand(-1, 3, -1, -1)
    diff_alpha_rgba = torch.cat([diff_alpha_rgb, union_alpha], dim=1)

    diff_rgb = torch.cat([diff_rgb, union_alpha], dim=1)  # Add alpha channel as 1s

    def to_image_grid(images):
        x = images.permute(0, 2, 3, 1)  # [B, H, W, C]
        return np.vstack(x.cpu().numpy())

    col1 = to_image_grid(target)
    col2 = to_image_grid(generated)
    col3 = to_image_grid(diff_rgb)
    col4 = to_image_grid(torch.cat([1.0 - target_alpha.expand(-1, 3, -1, -1), torch.ones_like(target_alpha)],
                                   dim=1))  # Add alpha channel as 1s
    col5 = to_image_grid(torch.cat([1.0 - generated_alpha.expand(-1, 3, -1, -1), torch.ones_like(generated_alpha)],
                                   dim=1))
    col6 = to_image_grid(diff_alpha_rgba)
    final_image = np.hstack((col1, col2, col3, col4, col5, col6))

    final_image = (final_image * 255.0).astype(np.uint8)  # Convert to uint8 for display
    return Image.fromarray(final_image).convert("RGBA")


if __name__ == '__main__':
    rf_loss_fn = RadianceFieldLoss(scene_path='../data/radiance_fields/lego', image_size=(512, 512),
                                   lpips_weight=0.0,
                                   device="cpu")

    camera, view_idx = rf_loss_fn.get_random_views(batch_size=3, mode='train')
    sampler = RaySampler((512, 512), mode='stride', stride=2)

    # print(rf_loss_fn.train_images_np.shape)
    # x = rf_loss_fn.train_images_np[0][..., :3]
    # x = Image.fromarray((x * 255).astype(np.uint8))
    # x.show()
    # print(rf_loss_fn.train_images[0])
    # exit()

    batch_size = 2
    x = torch.rand(batch_size, len(view_idx), 4, 256, 256)  # Simulated generated images [B, num_views, 4, H, W]
    alpha = torch.ones_like(x[:, :, 3:4])  # Assuming alpha channel is the 4th channel
    input_dict = {
        'generated_images_l1l2': x,
        'alpha_l1l2': alpha,
        'view_idx': view_idx,
        'mode': 'train',
        'sampler_l1l2': sampler
    }

    loss, loss_log, summary = rf_loss_fn(input_dict, return_summary=True)
    summary['images_l1l2'].show()
    # summary['images_lpips'].show()


## Model Helpers


In [ ]:
def precision_dtype(name: str):
    if name == 'float16' and device.type == 'cuda':
        return torch.float16
    return torch.float32


def make_grad_scaler(dtype):
    enabled = device.type == 'cuda' and dtype == torch.float16
    if hasattr(torch, 'amp'):
        return torch.amp.GradScaler(device.type, enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)


def normalize_model_grads(model):
    grads = [p.grad for p in model.parameters() if p.grad is not None]
    if not grads:
        return
    norm = torch.sqrt(sum(torch.sum(g.detach() ** 2) for g in grads))
    if torch.isfinite(norm) and norm > 0:
        for g in grads:
            g.div_(norm + 1e-8)


def make_models(cfg: RFNotebookConfig):
    if cfg.sh_degree != 0:
        raise NotImplementedError('This standalone notebook currently supports sh_degree=0, matching growing_rf.')

    dtype = precision_dtype(cfg.precision)
    nca = GrowingVNCA(
        channels=cfg.channels,
        fc_dim=cfg.fc_dim,
        noise_level=cfg.noise_level,
        update_prob=cfg.update_prob,
        seed_radius=cfg.seed_radius,
        padding=cfg.padding,
        precision=dtype,
        device=device,
    ).to(device)

    nca_output_dim = nca.channels
    if cfg.output_type == 'z':
        nca_output_dim *= nca.perception_kernels

    out_features = 1 + 3 * ((cfg.sh_degree + 1) ** 2)
    siren = Siren(
        in_features=nca_output_dim,
        coord_dim=3,
        hidden_features=cfg.hidden_features,
        hidden_layers=cfg.hidden_layers,
        out_features=out_features,
        outermost_linear=cfg.outermost_linear,
        fx=cfg.fx,
        activation=cfg.activation,
        num_frequencies=cfg.num_frequencies,
        first_omega_0=cfg.first_omega_0,
        hidden_omega_0=cfg.hidden_omega_0,
    ).to(device)
    return nca, siren, dtype


def make_renderer(cfg: RFNotebookConfig, dtype=torch.float32):
    return RendererRF(
        num_samples=cfg.num_samples,
        num_fine_samples=cfg.num_fine_samples,
        perturb=cfg.perturb,
        apply_living_mask=cfg.apply_living_mask,
        voxel_bounds=cfg.voxel_bounds,
        background_color=cfg.background_color,
        density_factor=cfg.density_factor,
        color_factor=cfg.color_factor,
        non_linearity=cfg.non_linearity,
        sh_degree=cfg.sh_degree,
        sampler_kwargs=copy.deepcopy(cfg.sampler_kwargs),
        precision=dtype,
    )


def make_rf_loss(cfg: RFNotebookConfig):
    return RadianceFieldLoss(
        scene_path=cfg.scene_path,
        image_size=cfg.image_size,
        l2_weight=cfg.l2_weight,
        l1_weight=cfg.l1_weight,
        lpips_weight=cfg.lpips_weight,
        device=device,
    )


## Train


In [ ]:
def train_growing_rf(cfg: RFNotebookConfig):
    set_seed(cfg.seed)
    nca, siren, dtype = make_models(cfg)
    renderer = make_renderer(cfg, dtype)
    rf_loss = make_rf_loss(cfg)

    params = list(nca.parameters()) + list(siren.parameters())
    optimizer = torch.optim.Adam(params, lr=cfg.lr)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=list(cfg.lr_decay_steps),
        gamma=cfg.lr_decay_gamma,
    )
    scaler = make_grad_scaler(dtype)

    pool = nca.seed(cfg.pool_size, *cfg.grid_size).detach()
    history = []

    for step in range(cfg.train_steps):
        idx = torch.randperm(cfg.pool_size, device=device)[:cfg.batch_size]
        x = pool[idx]
        if step % cfg.inject_seed_interval == 0:
            x[:1] = nca.seed(1, *cfg.grid_size)

        step_n = random.randrange(cfg.step_range[0], cfg.step_range[1])
        z = None
        with autocast_context(device, dtype):
            for _ in range(step_n):
                x, z = nca(x)

        x_render = (x if cfg.output_type == 's' else z).to(torch.float32)
        camera, view_idx = rf_loss.get_random_views(cfg.num_views, mode='train')

        rgb, depth, opacity, living_opacity, sampler = renderer.render(
            x_render,
            camera,
            siren,
            sampler_kwargs=copy.deepcopy(cfg.sampler_kwargs),
        )
        generated = torch.cat([rgb, opacity], dim=2)
        alpha_for_loss = living_opacity if living_opacity is not None else opacity

        input_dict = {
            'generated_images_l1l2': generated,
            'alpha_l1l2': alpha_for_loss,
            'view_idx': view_idx,
            'nca_state': x,
            'sampler_l1l2': sampler,
            'mode': 'train',
        }
        return_summary = (step % cfg.summary_interval == 0)
        loss, loss_log, summary = rf_loss(input_dict, return_summary=return_summary)
        overflow_loss = torch.relu(torch.abs(x) - 1.0).mean() * cfg.overflow_loss_weight
        total_loss = loss + overflow_loss

        optimizer.zero_grad(set_to_none=True)
        if dtype == torch.float16 and device.type == 'cuda':
            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            normalize_model_grads(nca)
            scaler.step(optimizer)
            scaler.update()
        else:
            total_loss.backward()
            normalize_model_grads(nca)
            optimizer.step()
        scheduler.step()

        pool[idx] = x.detach()

        row = {
            'step': step,
            'loss': float(total_loss.detach().cpu()),
            'rf_loss': float(loss.detach().cpu()),
            'overflow': float(overflow_loss.detach().cpu()),
            'psnr': float(loss_log.get('PSNR', torch.tensor(float('nan'))).detach().cpu()),
            'nca_steps': int(step_n),
        }
        history.append(row)

        if step % cfg.summary_interval == 0:
            print(row)

    return nca, siren, renderer, rf_loss, history


# Run after the scene data exists.
# nca, siren, renderer, rf_loss, history = train_growing_rf(cfg)


## Preview A Rollout


In [ ]:
@torch.no_grad()
def render_preview(nca, siren, renderer, cfg, steps=96, view_idx=0, mode='test'):
    scene_loss = make_rf_loss(cfg)
    cameras = scene_loss.test_cameras if mode == 'test' else scene_loss.train_cameras
    view_idx = int(np.clip(view_idx, 0, cameras.position.shape[0] - 1))
    camera = cameras.sample_batch(1, [view_idx])
    camera.height, camera.width = cfg.image_size

    x = nca.seed(1, *cfg.grid_size)
    z = None
    for _ in range(steps):
        x, z = nca(x)

    x_render = (x if cfg.output_type == 's' else z).float()
    rgb, depth, opacity, living, _ = renderer.render(
        x_render,
        camera,
        siren,
        batchify_rays=True,
        num_samples=max(cfg.num_samples, 128),
        num_fine_samples=max(cfg.num_fine_samples, 128),
        perturb=False,
        sampler_kwargs={'mode': 'stride', 'stride': 1},
    )
    img = renderer.to_pil((rgb, depth / 8.0, opacity, living))
    display(img)
    return img, x

# preview_img, preview_state = render_preview(nca, siren, renderer, cfg, steps=128)


## Optional Checkpoint


In [ ]:
def save_rf_checkpoint(path, nca, siren, cfg, history=None):
    path = Path(path)
    payload = {
        'cfg': asdict(cfg),
        'nca': nca.state_dict(),
        'siren': siren.state_dict(),
        'history': history or [],
    }
    torch.save(payload, path)
    print('saved checkpoint:', path)


def load_rf_checkpoint(path):
    payload = torch.load(path, map_location=device)
    loaded_cfg = RFNotebookConfig(**payload['cfg'])
    nca, siren, dtype = make_models(loaded_cfg)
    nca.load_state_dict(payload['nca'])
    siren.load_state_dict(payload['siren'])
    renderer = make_renderer(loaded_cfg, dtype)
    return nca, siren, renderer, loaded_cfg, payload.get('history', [])

# save_rf_checkpoint('growing_rf_notebook.pt', nca, siren, cfg, history)
# nca, siren, renderer, cfg, history = load_rf_checkpoint('growing_rf_notebook.pt')


## Final Export

Run the cell below after training. It writes the radiance-specific web package: `radiance_manifest.json`, `radiance_weights.bin`, and `radiance_weights.json`.

Default output folder: `demo/growing_radiance_fields/model/lego`


In [ ]:
try:
    import psutil
except ImportError:
    psutil = None


def _tensor_to_float32_array(tensor):
    return tensor.detach().cpu().contiguous().numpy().astype(np.float32)


class ExportableRadianceNCA:
    def __init__(self, nca, siren, cfg, iterations=0):
        self.nca = nca
        self.siren = siren
        self.cfg = cfg
        self.iterations = int(iterations)
        self.channels = int(nca.channels)
        self.perception_kernels = int(nca.perception_kernels)
        self.grid_size = tuple(int(v) for v in cfg.grid_size)
        self.living_channel = 3
        self.living_threshold = 0.1

    def state_dict(self):
        sd = OrderedDict()
        sd['perception.perceive.weight'] = self.nca.filters[:, None]
        sd['adaptation.adapt.0.weight'] = self.nca.w1.weight
        sd['adaptation.adapt.0.bias'] = self.nca.w1.bias
        sd['adaptation.adapt.2.weight'] = self.nca.w2.weight
        if self.nca.w2.bias is not None:
            sd['adaptation.adapt.2.bias'] = self.nca.w2.bias
        for name, tensor in self.siren.state_dict().items():
            sd[f'radiance.{name}'] = tensor
        return sd


def export_radiance_weights(
    rf_model,
    folder_path=None,
    json_filename='radiance_weights.json',
    bin_filename='radiance_weights.bin',
    manifest_filename='radiance_manifest.json',
):
    cfg = rf_model.cfg
    folder = Path(folder_path or cfg.export_dir)
    folder.mkdir(parents=True, exist_ok=True)

    json_path = folder / json_filename
    bin_path = folder / bin_filename
    manifest_path = folder / manifest_filename

    output_dim = 1 + 3 * ((cfg.sh_degree + 1) ** 2)
    manifest = {
        'format': 'cells2voxels-radiance-v1',
        'task': 'growing_rf',
        'standalone_notebook': True,
        'meta': {
            'experiment_name': cfg.experiment_name,
            'channels': int(rf_model.channels),
            'perception_kernels': int(rf_model.perception_kernels),
            'grid_size': list(rf_model.grid_size),
            'coarse_size': int(rf_model.grid_size[0]) if len(set(rf_model.grid_size)) == 1 else None,
            'living_channel': int(rf_model.living_channel),
            'living_threshold': float(rf_model.living_threshold),
            'seed_radius': int(cfg.seed_radius),
            'update_prob': float(cfg.update_prob),
            'padding': cfg.padding,
            'output_type': cfg.output_type,
            'iterations': int(rf_model.iterations),
            'precision': cfg.precision,
            'scene_path': cfg.scene_path,
            'gpu_used': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        },
        'renderer': {
            'type': 'radiance_field',
            'num_samples': int(cfg.num_samples),
            'num_fine_samples': int(cfg.num_fine_samples),
            'voxel_bounds': list(cfg.voxel_bounds),
            'background_color': float(cfg.background_color),
            'density_factor': float(cfg.density_factor),
            'color_factor': float(cfg.color_factor),
            'non_linearity': cfg.non_linearity,
            'apply_living_mask': bool(cfg.apply_living_mask),
            'sh_degree': int(cfg.sh_degree),
        },
        'siren': {
            'in_features': int(rf_model.channels if cfg.output_type == 's' else rf_model.channels * rf_model.perception_kernels),
            'coord_dim': 3,
            'out_features': int(output_dim),
            'hidden_features': int(cfg.hidden_features),
            'hidden_layers': int(cfg.hidden_layers),
            'outermost_linear': bool(cfg.outermost_linear),
            'fx': cfg.fx,
            'activation': cfg.activation,
            'num_frequencies': int(cfg.num_frequencies),
            'first_omega_0': float(cfg.first_omega_0),
            'hidden_omega_0': float(cfg.hidden_omega_0),
        },
        'perception': {},
        'adaptation': {},
        'radiance': {},
    }

    if psutil is not None:
        manifest['meta']['ram_used_gb'] = round(psutil.virtual_memory().used / (1024 ** 3), 2)

    byte_offset = 0
    with open(bin_path, 'wb') as bf:
        for full_name, tensor in rf_model.state_dict().items():
            arr = _tensor_to_float32_array(tensor)
            flat = arr.reshape(-1)
            if full_name.startswith('perception.'):
                section = 'perception'
                short_name = full_name.replace('perception.', '', 1)
            elif full_name.startswith('adaptation.'):
                section = 'adaptation'
                short_name = full_name.replace('adaptation.', '', 1)
            elif full_name.startswith('radiance.'):
                section = 'radiance'
                short_name = full_name.replace('radiance.', '', 1)
            else:
                print('Skipping non-exported tensor:', full_name)
                continue

            bf.write(flat.tobytes())
            manifest[section][short_name] = {
                'offset': int(byte_offset),
                'count': int(flat.size),
                'shape': list(arr.shape),
            }
            byte_offset += int(flat.nbytes)

    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2)

    index_path = folder.parent / 'index.json'
    export_name = folder.name
    try:
        names = json.loads(index_path.read_text(encoding='utf-8')) if index_path.exists() else []
        if export_name not in names:
            names.append(export_name)
        index_path.write_text(json.dumps(names, indent=2) + '\n', encoding='utf-8')
        print('Updated demo index      :', index_path)
    except Exception as exc:
        print('Could not update demo index:', exc)

    print('Exported runtime manifest:', manifest_path)
    print('Exported binary file     :', bin_path)
    print('Exported JSON copy       :', json_path)
    print('perception tensors:', len(manifest['perception']))
    print('adaptation tensors:', len(manifest['adaptation']))
    print('radiance tensors  :', len(manifest['radiance']))
    print('total float32 values:', byte_offset // 4)
    return json_path, bin_path, manifest_path


# Re-export only. No retraining needed once nca/siren/cfg exist.
# rf_export = ExportableRadianceNCA(nca, siren, cfg, iterations=cfg.train_steps)
# export_radiance_weights(rf_export, folder_path=cfg.export_dir)


In [ ]:
def inspect_radiance_export(folder=None):
    folder = Path(folder or cfg.export_dir)
    manifest = json.loads((folder / 'radiance_manifest.json').read_text(encoding='utf-8'))
    bin_size = (folder / 'radiance_weights.bin').stat().st_size
    indexed_bytes = 0
    for section in ['perception', 'adaptation', 'radiance']:
        for item in manifest[section].values():
            indexed_bytes += int(item['count']) * 4
    print('format:', manifest['format'])
    print('task:', manifest['task'])
    print('grid:', manifest['meta']['grid_size'])
    print('renderer:', manifest['renderer'])
    print('binary bytes:', bin_size)
    print('indexed bytes:', indexed_bytes)
    assert indexed_bytes == bin_size, 'Manifest byte counts do not match binary size'
    return manifest

# manifest = inspect_radiance_export(cfg.export_dir)
